# SAGIP-AI — Incident Photo Classifier (Colab)

Fine-tunes a small pretrained image model to sort incident photos into SAGIP-AI's 8 incident types,
then exports it to ONNX so the Next.js server can run it with no API and no daily quota.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`
2. Put `dataset.zip` in Google Drive at `MyDrive/sagip/dataset.zip`. Inside the zip:
```
dataset/
  fire/  flood/  landslide/  road_accident/
  structural_damage/  medical_emergency/  fallen_debris/  other/
```
Folder names must match `INCIDENT_TYPES` in `src/lib/constants.ts` exactly.

Run the cells top to bottom. Everything important is saved to `MyDrive/sagip/runs/<timestamp>/`,
because Colab deletes local files when the session ends.

## 1. Settings

In [ ]:
DRIVE_ZIP   = '/content/drive/MyDrive/sagip/dataset.zip'
OUT_ROOT    = '/content/drive/MyDrive/sagip/runs'
MODEL_NAME  = 'efficientnet_b0'   # or 'mobilenetv3_large_100' (smaller/faster)
IMG_SIZE    = 224
BATCH_SIZE  = 32
HEAD_EPOCHS = 5     # step 1: train only the new final layer
FULL_EPOCHS = 10    # step 2: fine-tune the whole model
SEED        = 42

# Must match src/lib/constants.ts INCIDENT_TYPES
EXPECTED_CLASSES = ['fire', 'flood', 'landslide', 'road_accident', 'structural_damage',
                    'medical_emergency', 'fallen_debris', 'other']

## 2. Mount Drive, unzip, install

In [ ]:
from google.colab import drive
import glob, os, zipfile
drive.mount('/content/drive')

# Find the dataset zip: the expected path first, then anywhere in Drive.
zip_path = DRIVE_ZIP if os.path.exists(DRIVE_ZIP) else None
if not zip_path:
    found = sorted(glob.glob('/content/drive/MyDrive/**/dataset*.zip', recursive=True))
    if not found:
        print('Top level of MyDrive:')
        for name in sorted(os.listdir('/content/drive/MyDrive'))[:40]:
            print(' ', name)
        raise FileNotFoundError(
            'No dataset zip found. Expected ' + DRIVE_ZIP +
            '. Upload dataset.zip to Drive and wait for the upload to finish,'
            ' or set DRIVE_ZIP in the settings cell to its real path.')
    zip_path = found[0]
    print('Using', zip_path, '(not the expected path)')

size_mb = os.path.getsize(zip_path) / 1e6
print('Zip:', zip_path, '(%.0f MB)' % size_mb)
if size_mb < 1:
    raise ValueError('The zip is nearly empty, so the upload probably has not finished. Wait, then re-run this cell.')

!rm -rf /content/dataset
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/')
    print('Extracted', len(z.namelist()), 'entries')

# The class folders may sit one level deeper, depending on how the zip was made.
if not os.path.isdir('/content/dataset'):
    candidates = [d for d in glob.glob('/content/*/') if any(os.path.isdir(d + c) for c in EXPECTED_CLASSES)]
    if not candidates:
        print('In /content:', os.listdir('/content'))
        raise FileNotFoundError('Extracted, but no folder with the expected class subfolders was found.')
    os.rename(candidates[0].rstrip('/'), '/content/dataset')
print('Class folders:', sorted(os.listdir('/content/dataset')))

!pip install -q timm onnx onnxscript onnxruntime scikit-learn

In [ ]:
import os, json, time, random, collections
import numpy as np, torch, timm
from PIL import Image, ImageFile
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '(switch the runtime to T4 GPU if this says cpu)')

RUN_DIR = os.path.join(OUT_ROOT, time.strftime('%Y%m%d-%H%M%S'))
os.makedirs(RUN_DIR, exist_ok=True)
print('Saving to', RUN_DIR)

## 3. Check the dataset

In [ ]:
DATA_DIR = '/content/dataset'
full = datasets.ImageFolder(DATA_DIR)
classes = full.classes  # alphabetical order = the model's output order
missing = sorted(set(EXPECTED_CLASSES) - set(classes))
extra = sorted(set(classes) - set(EXPECTED_CLASSES))
assert not extra, f'Unknown folders (fix the names): {extra}'
if missing: print('WARNING: no folder for', missing, '- the model will never predict these')

counts = collections.Counter(full.targets)
for i, c in enumerate(classes):
    flag = '  <- too few, aim for 100+' if counts[i] < 100 else ''
    print(f'{c:20s} {counts[i]:5d}{flag}')
print('Total:', len(full))

## 4. Split 70 / 15 / 15 (stratified)
The **test set is locked**: never tune on it. Its score is the honest number for your thesis.

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(full)); y = np.array(full.targets)
tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=y[tmp_idx], random_state=SEED)
print(len(tr_idx), 'train /', len(va_idx), 'val /', len(te_idx), 'test')

# Save the exact test file list so the same photos can be run through the Gemini benchmark.
test_files = [os.path.relpath(full.samples[i][0], DATA_DIR) for i in te_idx]
with open(os.path.join(RUN_DIR, 'test_files.json'), 'w') as f: json.dump(test_files, f, indent=1)

## 5. Augmentation and loaders

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.3),   # low light, bad phone cameras
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),  # motion blur / rain
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 256 / 224)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class Subset(Dataset):
    def __init__(self, indices, tf): self.indices, self.tf = indices, tf
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        path, label = full.samples[self.indices[i]]
        return self.tf(Image.open(path).convert('RGB')), label

train_dl = DataLoader(Subset(tr_idx, train_tf), BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(Subset(va_idx, eval_tf), BATCH_SIZE, num_workers=2)
test_dl  = DataLoader(Subset(te_idx, eval_tf), BATCH_SIZE, num_workers=2)

## 6. Model and training loop

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(classes)).to(device)

# Class weights so rare types (e.g. landslide) aren't ignored.
tr_counts = np.bincount(y[tr_idx], minlength=len(classes)).astype(float)
weights = torch.tensor(tr_counts.sum() / (len(classes) * np.maximum(tr_counts, 1)), dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

def run_epoch(dl, opt=None):
    model.train(opt is not None)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(opt is not None):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb); loss = loss_fn(out, yb)
            if opt: opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(yb); total += len(yb)
            correct += (out.argmax(1) == yb).sum().item()
    return loss_sum / total, correct / total

best_acc, best_path = 0.0, os.path.join(RUN_DIR, 'best.pt')
def train(epochs, lr, stage):
    global best_acc
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for ep in range(1, epochs + 1):
        tl, ta = run_epoch(train_dl, opt); vl, va = run_epoch(val_dl); sched.step()
        mark = ''
        if va > best_acc:
            best_acc = va; torch.save(model.state_dict(), best_path); mark = '  * saved'
        print(f'[{stage}] epoch {ep:2d}  train acc {ta:.3f}  val acc {va:.3f}  val loss {vl:.3f}{mark}')

### Step 1: freeze the pretrained layers, train only the new head

In [ ]:
for p in model.parameters(): p.requires_grad = False
for p in model.get_classifier().parameters(): p.requires_grad = True
train(HEAD_EPOCHS, 1e-3, 'head')

### Step 2: unfreeze everything, fine-tune at a low learning rate

In [ ]:
for p in model.parameters(): p.requires_grad = True
train(FULL_EPOCHS, 1e-4, 'full')
print('Best val acc:', round(best_acc, 3))

## 7. Evaluate on the locked test set
Accuracy, per-class results, confusion matrix and calibration. Put these numbers in your thesis.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
model.load_state_dict(torch.load(best_path)); model.eval()
probs, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        probs.append(torch.softmax(model(xb.to(device)), 1).cpu()); labels.append(yb)
probs = torch.cat(probs).numpy(); labels = torch.cat(labels).numpy()
pred, conf = probs.argmax(1), probs.max(1)

print(f'Test accuracy: {(pred == labels).mean():.3f}  ({len(labels)} photos)\n')
print(classification_report(labels, pred, labels=range(len(classes)), target_names=classes, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
cm = confusion_matrix(labels, pred, labels=range(len(classes)))
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right'); ax.set_yticks(range(len(classes)), classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('Expected'); ax.set_title('Confusion matrix (test set)')
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, 'confusion_matrix.png'), dpi=150); plt.show()

In [ ]:
# Calibration: when the model says 90% sure, is it right ~90% of the time?
# SAGIP-AI routes reports below CONFIDENCE_THRESHOLD (default 0.7) to human review.
bins = np.linspace(0, 1, 6)
print('confidence    n   accuracy')
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
    if m.any(): print(f'{lo:.1f}-{hi:.1f}  {m.sum():5d}   {(pred[m] == labels[m]).mean():.3f}')
auto = conf >= 0.7
print(f'\nAt threshold 0.7: {auto.mean():.1%} auto-classified, accuracy on those {(pred[auto] == labels[auto]).mean() if auto.any() else 0:.3f}')
brier = np.mean((conf - (pred == labels)) ** 2)
print(f'Brier score (lower is better, same metric as the benchmark): {brier:.3f}')

json.dump({'model': MODEL_NAME, 'test_accuracy': float((pred == labels).mean()), 'brier': float(brier),
           'best_val_accuracy': float(best_acc), 'n_test': int(len(labels))},
          open(os.path.join(RUN_DIR, 'metrics.json'), 'w'), indent=2)

## 8. Export to ONNX
Produces the two files the SAGIP-AI server needs: `sagip-classifier.onnx` and `labels.json`.

In [ ]:
model.eval().cpu()
onnx_path = os.path.join(RUN_DIR, 'sagip-classifier.onnx')
torch.onnx.export(model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), onnx_path,
                  input_names=['image'], output_names=['logits'],
                  dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
json.dump({'classes': classes, 'image_size': IMG_SIZE, 'mean': MEAN, 'std': STD, 'model': MODEL_NAME},
          open(os.path.join(RUN_DIR, 'labels.json'), 'w'), indent=2)
print('Size:', round(os.path.getsize(onnx_path) / 1e6, 1), 'MB')

### Check the ONNX file gives the same answers as PyTorch

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
xb, _ = next(iter(test_dl))
with torch.no_grad(): torch_out = model(xb).numpy()
onnx_out = sess.run(None, {'image': xb.numpy()})[0]
print('Max difference:', float(np.abs(torch_out - onnx_out).max()), '(should be < 0.001)')
print('Same predictions:', bool((torch_out.argmax(1) == onnx_out.argmax(1)).all()))

## Done
Everything is in `MyDrive/sagip/runs/<timestamp>/`:
- `sagip-classifier.onnx` and `labels.json`, the files for SAGIP-AI's `local` provider
- `metrics.json`, `confusion_matrix.png` and `test_files.json` for your thesis and the benchmark comparison

**If test accuracy is below ~80%:** check the confusion matrix, then add more photos to the classes that get mixed up.
More data helps much more than changing settings.